# T1 - Project Setup and Final Stack Lock

**Guardian Eye task:** T1

Run this notebook top-to-bottom on the target virtual machine. Configuration is centralized near the top, and heavy model work only occurs in explicitly marked execution cells.


## 1. Locked stack and paths

This cell declares the offline stack selected by the design document. Change model paths to match the target VM before running setup.


In [ ]:
from dataclasses import dataclass, asdict
from pathlib import Path
import json

@dataclass(frozen=True)
class StackLock:
    python: str = ">=3.10,<3.13"
    classifier: str = "V9 RLVS (existing local checkpoint)"
    visual_narrator: str = "Qwen2.5-VL-3B-Instruct-AWQ"
    text_model: str = "Qwen2.5/Qwen3 local instruct model, selected by available VRAM"
    embeddings: str = "BAAI/bge-small-en-v1.5"
    vector_store: str = "faiss-cpu"
    metadata_store: str = "SQLite"
    api: str = "FastAPI + Uvicorn"
    runtime: str = "Fully local/offline; heavy models load on demand"

ROOT = Path.cwd()
MODEL_PATHS = {
    "classifier": ROOT / "models" / "v9_rlvs",
    "vlm": ROOT / "models" / "qwen2.5-vl-3b-instruct-awq",
    "text_llm": ROOT / "models" / "text_llm",
    "embeddings": ROOT / "models" / "bge-small-en-v1.5",
}

print(json.dumps(asdict(StackLock()), indent=2))
print({name: str(value) for name, value in MODEL_PATHS.items()})


## 2. Create the repository layout

Creates only missing directories. It does not overwrite code from other team members.


In [ ]:
DIRECTORIES = [
    "backend/api", "backend/core", "backend/schemas",
    "models/classifier", "models/narrator", "models/text_llm", "models/embeddings",
    "rag/reference_corpus", "rag/incident_memory",
    "data/incoming", "data/telemetry", "data/frames", "data/recaps",
    "storage/faiss", "storage/sqlite", "frontend", "scripts", "tests", "docs", "logs",
]

for relative in DIRECTORIES:
    (ROOT / relative).mkdir(parents=True, exist_ok=True)

print(f"Ensured {len(DIRECTORIES)} directories under {ROOT}")


## 3. Write dependency lock inputs

Writes a practical requirements file for the integrated system. Install the CUDA-compatible PyTorch build separately if the VM requires a specific CUDA wheel.


In [ ]:
requirements = """# API and schemas
fastapi>=0.115,<1
uvicorn[standard]>=0.34,<1
pydantic>=2.10,<3
python-multipart>=0.0.20,<1

# Models and numeric runtime
torch>=2.5,<3
torchvision>=0.20,<1
transformers>=4.49,<5
accelerate>=1.3,<2
autoawq>=0.2.7,<1
sentence-transformers>=3.4,<4
numpy>=1.26,<3
pillow>=10,<12
opencv-python-headless>=4.10,<5

# Local retrieval and persistence
faiss-cpu>=1.9,<2
sqlalchemy>=2.0,<3

# Tests and notebooks
pytest>=8.3,<9
httpx>=0.28,<1
jupyterlab>=4.3,<5
"""

(ROOT / "requirements.txt").write_text(requirements, encoding="utf-8")
print("Wrote requirements.txt")


## 4. Write environment template

Creates a `.env.example` containing paths and runtime controls. The backend should read these values, but the template contains no secrets.


In [ ]:
env_example = """GUARDIAN_EYE_OFFLINE=1
CLASSIFIER_ADAPTER=models.classifier.v9_adapter:load_classifier
CLASSIFIER_MODEL_PATH=models/v9_rlvs
VLM_MODEL_PATH=models/qwen2.5-vl-3b-instruct-awq
TEXT_LLM_MODEL_PATH=models/text_llm
EMBEDDING_MODEL_PATH=models/bge-small-en-v1.5
INCIDENT_DB_PATH=storage/sqlite/incidents.db
REFERENCE_INDEX_PATH=storage/faiss/reference.index
INCIDENT_INDEX_PATH=storage/faiss/incidents.index
FRAME_OUTPUT_DIR=data/frames
RECAP_OUTPUT_DIR=data/recaps
LOG_LEVEL=INFO
"""
(ROOT / ".env.example").write_text(env_example, encoding="utf-8")
print("Wrote .env.example")


## 5. Write the integration README

Documents ownership boundaries, contracts, setup, and run commands for the team.


In [ ]:
readme = """# Guardian Eye

Fully local RAG-assisted violence-analysis demo.

## Authority rule
The V9 RLVS classifier is the sole authority for `verdict` and `confidence`.
The VLM narrates evidence and must never re-decide either value.

## Setup
1. Create and activate a Python 3.10+ virtual environment.
2. Install the CUDA-compatible PyTorch build for the VM.
3. Run `pip install -r requirements.txt`.
4. Copy `.env.example` to `.env` and update local model paths.
5. Place all checkpoints locally. The demo must not make cloud calls.

## Module ownership
- Tera: classifier adapter, orchestration API, sequential runtime, integration tests.
- 3elwa: deterministic evidence packet, reference retrieval, incident memory/search.
- elweeka: VLM/LLM narration, guardrails, recap formatting.

## Run
- API: `uvicorn backend.api.main:app --host 0.0.0.0 --port 8000`
- Tests: `pytest -q tests`
- Demo: `python scripts/demo_run.py --video data/incoming/example.mp4`

## Required integration contracts
Classifier output: verdict, confidence, threshold, gate[4], gqs[5], telemetry,
frames_vit[16,224,224,3], source, timestamp.

Analyze response: verdict, confidence, evidence_packet, explanation, incident_id,
recap_frame_paths, limitations_note.
"""
(ROOT / "README.md").write_text(readme, encoding="utf-8")
print("Wrote README.md")


## 6. Static setup check

Checks that setup artifacts and model-path configuration exist without loading any model.


In [ ]:
required = [ROOT / "requirements.txt", ROOT / ".env.example", ROOT / "README.md"]
missing = [str(p) for p in required if not p.exists()]
assert not missing, f"Missing setup artifacts: {missing}"

for name, model_path in MODEL_PATHS.items():
    print(f"{name:12} configured={model_path} exists={model_path.exists()}")

print("Project setup artifacts are present.")
